# Dataset Exploration - YouTube Thumbnail Predictor

This notebook explores the scraped YouTube video dataset to understand the data distribution and quality.

## 1. Load Data

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
import os
import json

# Set style
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

# Load the scored dataset (with Vision API labels and thumbnail scores)
df = pd.read_csv('../data/videos_scored_v3.csv')

print(f"Dataset shape: {df.shape}")
print(f"\nColumns: {df.columns.tolist()}")
df.head()

In [ ]:
# Data types and basic info
df.info()

## 2. Channel Statistics

In [ ]:
# Videos per channel
channel_counts = df['channel_name'].value_counts()

fig, ax = plt.subplots(figsize=(14, 6))
bars = ax.bar(range(len(channel_counts)), channel_counts.values, color='steelblue')
ax.set_xticks(range(len(channel_counts)))
ax.set_xticklabels(channel_counts.index, rotation=45, ha='right')
ax.set_xlabel('Channel')
ax.set_ylabel('Number of Videos')
ax.set_title('Videos per Channel')

# Add value labels on bars
for bar, count in zip(bars, channel_counts.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2, 
            str(count), ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.show()

print(f"Total channels: {df['channel_name'].nunique()}")
print(f"Total videos: {len(df)}")

In [ ]:
# Videos per niche
niche_counts = df['niche'].value_counts()

fig, ax = plt.subplots(figsize=(8, 5))
colors = ['#3498db', '#e74c3c']
bars = ax.bar(niche_counts.index, niche_counts.values, color=colors)
ax.set_xlabel('Niche')
ax.set_ylabel('Number of Videos')
ax.set_title('Videos per Niche')

for bar, count in zip(bars, niche_counts.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5, 
            str(count), ha='center', va='bottom', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.show()

## 3. Performance Distribution

In [ ]:
# Performance score statistics
print("Performance Score Statistics:")
print(df['performance_score'].describe())

In [ ]:
# Histogram of performance scores
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Raw distribution
axes[0].hist(df['performance_score'], bins=50, color='steelblue', edgecolor='black', alpha=0.7)
axes[0].set_xlabel('Performance Score')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Performance Score Distribution')
axes[0].axvline(df['performance_score'].median(), color='red', linestyle='--', label=f"Median: {df['performance_score'].median():.4f}")
axes[0].legend()

# Log-scaled distribution for better visibility
axes[1].hist(df['performance_score'], bins=50, color='steelblue', edgecolor='black', alpha=0.7)
axes[1].set_xlabel('Performance Score')
axes[1].set_ylabel('Frequency (log scale)')
axes[1].set_title('Performance Score Distribution (Log Scale)')
axes[1].set_yscale('log')

plt.tight_layout()
plt.show()

In [ ]:
# Performance by niche
fig, ax = plt.subplots(figsize=(10, 6))
df.boxplot(column='performance_score', by='niche', ax=ax)
ax.set_xlabel('Niche')
ax.set_ylabel('Performance Score')
ax.set_title('Performance Score by Niche')
plt.suptitle('')  # Remove automatic title
plt.tight_layout()
plt.show()

## 4. Top Performers

In [ ]:
# Top 20 performing videos
top_performers = df.nlargest(20, 'performance_score')[[
    'title', 'channel_name', 'niche', 'view_count', 'subscriber_count', 'performance_score'
]].copy()

# Format numbers for readability
top_performers['view_count'] = top_performers['view_count'].apply(lambda x: f"{x:,}")
top_performers['subscriber_count'] = top_performers['subscriber_count'].apply(lambda x: f"{x:,}")
top_performers['performance_score'] = top_performers['performance_score'].apply(lambda x: f"{x:.4f}")

# Truncate long titles
top_performers['title'] = top_performers['title'].apply(lambda x: x[:50] + '...' if len(x) > 50 else x)

top_performers

In [ ]:
# View count vs performance score scatter plot
fig, ax = plt.subplots(figsize=(12, 6))

scatter = ax.scatter(df['view_count'], df['performance_score'], 
                     c=df['niche'].map({'AI/Tech': 0, 'Business': 1}), 
                     cmap='coolwarm', alpha=0.6, s=30)

ax.set_xlabel('View Count')
ax.set_ylabel('Performance Score')
ax.set_title('View Count vs Performance Score')
ax.set_xscale('log')

# Add legend
handles = [plt.scatter([], [], c='blue', label='AI/Tech'),
           plt.scatter([], [], c='red', label='Business')]
ax.legend(handles=handles)

plt.tight_layout()
plt.show()

## 5. Thumbnail Gallery

In [ ]:
def show_thumbnail_grid(dataframe, n_rows=3, n_cols=4, title="Sample Thumbnails", score_col='performance_score'):
    """Display a grid of thumbnails from the dataset."""
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(16, n_rows * 3))
    fig.suptitle(title, fontsize=14, fontweight='bold')
    
    for idx, ax in enumerate(axes.flat):
        if idx < len(dataframe):
            row = dataframe.iloc[idx]
            thumbnail_path = f"../data/thumbnails/{row['video_id']}.jpg"
            
            if os.path.exists(thumbnail_path):
                img = Image.open(thumbnail_path)
                ax.imshow(img)
                if score_col == 'thumbnail_score_v3':
                    ax.set_title(f"{row['channel_name']}\nScore: {row[score_col]:.1f}/20", fontsize=9)
                else:
                    ax.set_title(f"{row['channel_name']}\nPerf: {row[score_col]:.4f}", fontsize=9)
            else:
                ax.text(0.5, 0.5, 'Not found', ha='center', va='center')
        ax.axis('off')
    
    plt.tight_layout()
    plt.show()

In [ ]:
# Show top performing thumbnails
top_videos = df.nlargest(12, 'performance_score')
show_thumbnail_grid(top_videos, n_rows=3, n_cols=4, title="Top Performing Thumbnails")

In [ ]:
# Show random sample of thumbnails
random_sample = df.sample(12, random_state=42)
show_thumbnail_grid(random_sample, n_rows=3, n_cols=4, title="Random Sample of Thumbnails")

In [ ]:
# Show lowest performing thumbnails
bottom_videos = df.nsmallest(12, 'performance_score')
show_thumbnail_grid(bottom_videos, n_rows=3, n_cols=4, title="Lowest Performing Thumbnails")

## 6. Data Quality

In [ ]:
# Check for missing values
print("Missing Values:")
missing = df.isnull().sum()
print(missing[missing > 0] if missing.any() else "No missing values found!")

print(f"\nTotal missing values: {df.isnull().sum().sum()}")

In [ ]:
# Check for duplicates
duplicates = df.duplicated(subset=['video_id']).sum()
print(f"Duplicate video IDs: {duplicates}")

title_duplicates = df.duplicated(subset=['title']).sum()
print(f"Duplicate titles: {title_duplicates}")

In [ ]:
# Check thumbnail files exist
thumbnail_dir = '../data/thumbnails/'
missing_thumbnails = []

for video_id in df['video_id']:
    thumbnail_path = os.path.join(thumbnail_dir, f"{video_id}.jpg")
    if not os.path.exists(thumbnail_path):
        missing_thumbnails.append(video_id)

print(f"Videos in CSV: {len(df)}")
print(f"Thumbnails on disk: {len(os.listdir(thumbnail_dir))}")
print(f"Missing thumbnails: {len(missing_thumbnails)}")

if missing_thumbnails:
    print(f"\nMissing thumbnail IDs: {missing_thumbnails[:10]}...")

In [ ]:
# Data summary
print("=" * 60)
print("DATASET SUMMARY")
print("=" * 60)
print(f"Total videos: {len(df):,}")
print(f"Total channels: {df['channel_name'].nunique()}")
print(f"Total niches: {df['niche'].nunique()}")
print(f"\nDate range: {df['published_at'].min()} to {df['published_at'].max()}")
print(f"\nView count range: {df['view_count'].min():,} to {df['view_count'].max():,}")
print(f"Performance score range: {df['performance_score'].min():.6f} to {df['performance_score'].max():.6f}")
print("=" * 60)

---

## 7. Vision API Features Analysis

In [ ]:
# Vision API feature overview
print("=" * 60)
print("VISION API FEATURES OVERVIEW")
print("=" * 60)

print(f"\nFace Detection:")
print(f"  Faces present: {df['visage_present'].sum()} ({df['visage_present'].mean()*100:.1f}%)")
print(f"  Number of people distribution:")
print(df['nb_personnes'].value_counts().sort_index())

print(f"\nText Detection:")
print(f"  Text present: {df['texte_present'].sum()} ({df['texte_present'].mean()*100:.1f}%)")

print(f"\nExpression Distribution:")
print(df['expression'].value_counts())

print(f"\nColor Distribution:")
print(df['couleur_dominante'].value_counts())

print(f"\nBackground Type:")
print(df['fond'].value_counts())

In [ ]:
# Vision features visualization
fig, axes = plt.subplots(2, 3, figsize=(16, 10))

# 1. Face presence
face_counts = df['visage_present'].value_counts()
axes[0, 0].pie(face_counts.values, labels=['Face', 'No Face'], autopct='%1.1f%%', colors=['#3498db', '#e74c3c'])
axes[0, 0].set_title('Face Presence')

# 2. Text presence
text_counts = df['texte_present'].value_counts()
axes[0, 1].pie(text_counts.values, labels=['Text', 'No Text'], autopct='%1.1f%%', colors=['#2ecc71', '#e74c3c'])
axes[0, 1].set_title('Text Presence')

# 3. Number of people
people_counts = df['nb_personnes'].value_counts().sort_index()
axes[0, 2].bar(people_counts.index.astype(str), people_counts.values, color='steelblue')
axes[0, 2].set_xlabel('Number of People')
axes[0, 2].set_ylabel('Count')
axes[0, 2].set_title('Number of People in Thumbnail')

# 4. Expression
expr_counts = df['expression'].value_counts()
axes[1, 0].barh(expr_counts.index, expr_counts.values, color='#9b59b6')
axes[1, 0].set_xlabel('Count')
axes[1, 0].set_title('Expression Distribution')

# 5. Color palette
color_counts = df['couleur_dominante'].value_counts()
color_map = {'Neutre': '#95a5a6', 'Chaud': '#e74c3c', 'Froid': '#3498db'}
axes[1, 1].bar(color_counts.index, color_counts.values, color=[color_map.get(c, 'gray') for c in color_counts.index])
axes[1, 1].set_xlabel('Color Palette')
axes[1, 1].set_ylabel('Count')
axes[1, 1].set_title('Dominant Color Distribution')

# 6. Background type
bg_counts = df['fond'].value_counts()
axes[1, 2].bar(bg_counts.index, bg_counts.values, color='#1abc9c')
axes[1, 2].set_xlabel('Background Type')
axes[1, 2].set_ylabel('Count')
axes[1, 2].set_title('Background Type Distribution')

plt.tight_layout()
plt.show()

In [ ]:
# Feature correlation with performance
print("=" * 60)
print("FEATURE IMPACT ON PERFORMANCE")
print("=" * 60)

print("\nAverage performance by feature value:")
print("\n--- Face Presence ---")
print(df.groupby('visage_present')['performance_score'].mean())

print("\n--- Text Presence ---")
print(df.groupby('texte_present')['performance_score'].mean())

print("\n--- Number of People ---")
print(df.groupby('nb_personnes')['performance_score'].mean().sort_values(ascending=False))

print("\n--- Expression ---")
print(df.groupby('expression')['performance_score'].mean().sort_values(ascending=False))

print("\n--- Background Type ---")
print(df.groupby('fond')['performance_score'].mean().sort_values(ascending=False))

---

## 8. Thumbnail Score Analysis (ML-based)

In [ ]:
# Load model weights
with open('../data/thumbnail_weights_v3.json', 'r') as f:
    weights = json.load(f)

print("=" * 60)
print("THUMBNAIL SCORING MODEL")
print("=" * 60)
print(f"\nModel: {weights.get('model_type', 'Unknown')}")
print(f"ROC-AUC: {weights.get('roc_auc', 0):.4f}")
print(f"Top Performer Threshold: {weights.get('threshold_top_performer', 0):.6f}")

In [ ]:
# Feature importance visualization
feature_importance = {k: v for k, v in weights.items() 
                      if k not in ['model_type', 'roc_auc', 'threshold_top_performer']}

importance_df = pd.DataFrame({
    'feature': feature_importance.keys(),
    'importance': feature_importance.values()
}).sort_values('importance', ascending=True)

fig, ax = plt.subplots(figsize=(10, 8))
colors = ['#e74c3c' if x < importance_df['importance'].median() else '#2ecc71' 
          for x in importance_df['importance']]
ax.barh(importance_df['feature'], importance_df['importance'], color=colors)
ax.set_xlabel('Feature Importance')
ax.set_title('Thumbnail Scoring - Feature Importance')
ax.axvline(x=importance_df['importance'].median(), color='gray', linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()

In [ ]:
# Thumbnail score distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram
axes[0].hist(df['thumbnail_score_v3'], bins=20, color='#3498db', edgecolor='black', alpha=0.7)
axes[0].set_xlabel('Thumbnail Score')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Thumbnail Score Distribution (/20)')
axes[0].axvline(df['thumbnail_score_v3'].mean(), color='red', linestyle='--', 
                label=f"Mean: {df['thumbnail_score_v3'].mean():.1f}")
axes[0].axvline(df['thumbnail_score_v3'].median(), color='green', linestyle='--', 
                label=f"Median: {df['thumbnail_score_v3'].median():.1f}")
axes[0].legend()

# Score bands
score_bands = pd.cut(df['thumbnail_score_v3'], bins=[0, 5, 10, 15, 20], labels=['0-5', '5-10', '10-15', '15-20'])
band_counts = score_bands.value_counts().sort_index()
colors = ['#e74c3c', '#f39c12', '#3498db', '#2ecc71']
axes[1].bar(band_counts.index, band_counts.values, color=colors)
axes[1].set_xlabel('Score Band')
axes[1].set_ylabel('Count')
axes[1].set_title('Thumbnails by Score Band')

for i, (band, count) in enumerate(band_counts.items()):
    axes[1].text(i, count + 10, str(count), ha='center', fontweight='bold')

plt.tight_layout()
plt.show()

print(f"\nScore Statistics:")
print(df['thumbnail_score_v3'].describe())

In [ ]:
# Score vs Performance validation
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Scatter plot
axes[0].scatter(df['thumbnail_score_v3'], df['performance_score'], alpha=0.5, s=20)
axes[0].set_xlabel('Thumbnail Score (/20)')
axes[0].set_ylabel('Performance Score')
axes[0].set_title(f'Thumbnail Score vs Performance\n(Correlation: {df["thumbnail_score_v3"].corr(df["performance_score"]):.4f})')

# Performance by score band
df['score_band'] = pd.cut(df['thumbnail_score_v3'], bins=[0, 5, 10, 15, 20], labels=['0-5', '5-10', '10-15', '15-20'])
band_perf = df.groupby('score_band')['performance_score'].mean()
axes[1].bar(band_perf.index, band_perf.values, color=['#e74c3c', '#f39c12', '#3498db', '#2ecc71'])
axes[1].set_xlabel('Score Band')
axes[1].set_ylabel('Average Performance Score')
axes[1].set_title('Average Performance by Score Band')

plt.tight_layout()
plt.show()

# Detailed breakdown
print("\nPerformance by Score Band:")
print("="*50)
for band in ['0-5', '5-10', '10-15', '15-20']:
    subset = df[df['score_band'] == band]
    if len(subset) > 0:
        top_pct = (subset['performance_score'] >= weights.get('threshold_top_performer', 0.00177)).mean() * 100
        print(f"{band:8s}: n={len(subset):4d}, avg_perf={subset['performance_score'].mean():.6f}, top_performer%={top_pct:.1f}%")

In [ ]:
# Thumbnail gallery by score band
print("\n" + "="*60)
print("THUMBNAIL GALLERY BY SCORE")
print("="*60)

In [ ]:
# Highest scoring thumbnails
top_scored = df.nlargest(12, 'thumbnail_score_v3')
show_thumbnail_grid(top_scored, n_rows=3, n_cols=4, 
                    title="Highest Scoring Thumbnails (15-20)", 
                    score_col='thumbnail_score_v3')

In [ ]:
# Medium scoring thumbnails (10-15)
medium_scored = df[(df['thumbnail_score_v3'] >= 10) & (df['thumbnail_score_v3'] < 15)].sample(min(12, len(df[(df['thumbnail_score_v3'] >= 10) & (df['thumbnail_score_v3'] < 15)])), random_state=42)
show_thumbnail_grid(medium_scored, n_rows=3, n_cols=4, 
                    title="Medium Scoring Thumbnails (10-15)", 
                    score_col='thumbnail_score_v3')

In [ ]:
# Lowest scoring thumbnails
low_scored = df.nsmallest(12, 'thumbnail_score_v3')
show_thumbnail_grid(low_scored, n_rows=3, n_cols=4, 
                    title="Lowest Scoring Thumbnails (0-5)", 
                    score_col='thumbnail_score_v3')

In [ ]:
# Score by channel
channel_scores = df.groupby('channel_name').agg({
    'thumbnail_score_v3': 'mean',
    'performance_score': 'mean',
    'video_id': 'count'
}).rename(columns={'video_id': 'video_count'}).sort_values('thumbnail_score_v3', ascending=False)

fig, ax = plt.subplots(figsize=(14, 6))
bars = ax.bar(range(len(channel_scores)), channel_scores['thumbnail_score_v3'].values, color='steelblue')
ax.set_xticks(range(len(channel_scores)))
ax.set_xticklabels(channel_scores.index, rotation=45, ha='right')
ax.set_xlabel('Channel')
ax.set_ylabel('Average Thumbnail Score (/20)')
ax.set_title('Average Thumbnail Score by Channel')
ax.axhline(y=df['thumbnail_score_v3'].mean(), color='red', linestyle='--', label=f"Overall Mean: {df['thumbnail_score_v3'].mean():.1f}")
ax.legend()

plt.tight_layout()
plt.show()

print("\nChannel Scores:")
print(channel_scores.round(2))

---

## 9. Score Interpretation Guide

In [ ]:
print("""
╔══════════════════════════════════════════════════════════════════╗
║                 THUMBNAIL SCORE INTERPRETATION                    ║
╠══════════════════════════════════════════════════════════════════╣
║                                                                  ║
║  Score 0-5:   POOR - Missing key elements                       ║
║               • Low probability of top performance              ║
║               • Consider adding text, faces, or better colors   ║
║                                                                  ║
║  Score 5-10:  BELOW AVERAGE - Some elements present             ║
║               • ~33% chance of being a top performer            ║
║               • Room for improvement in composition             ║
║                                                                  ║
║  Score 10-15: GOOD - Most best practices followed               ║
║               • ~73% chance of being a top performer            ║
║               • Solid thumbnail with minor optimizations needed ║
║                                                                  ║
║  Score 15-20: EXCELLENT - High probability of top performance   ║
║               • 100% of these are top performers in our data    ║
║               • Optimal combination of visual elements          ║
║                                                                  ║
╠══════════════════════════════════════════════════════════════════╣
║  KEY FACTORS (by importance):                                    ║
║  1. Cold colors (blue tones)                                     ║
║  2. High face detection confidence                               ║
║  3. Text presence and length                                     ║
║  4. Number of people (1-2 optimal)                               ║
║  5. Background complexity                                        ║
╚══════════════════════════════════════════════════════════════════╝
""")